### RAG_pipeline - Data Ingestion to Vector DB pipeline

In [9]:
import os 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
# function to read all the document or pdf 

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"   Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"   Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: Ensuring Success in Job Interviews.pdf
   Loaded 2 pages

Processing: Factors Responsible for Success.pdf
   Loaded 3 pages

Processing: How To Prepare for a Job Interview.pdf
   Loaded 4 pages

Processing: Interview Dress Code.pdf
   Loaded 8 pages

Processing: Power Dressing for Men.pdf
   Loaded 2 pages

Total documents loaded: 19


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-12-14T04:29:50+00:00', 'author': 'Huma  Iqbal', 'moddate': '2022-12-14T04:29:50+00:00', 'source': '..\\data\\pdf\\Ensuring Success in Job Interviews.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Ensuring Success in Job Interviews.pdf', 'file_type': 'pdf'}, page_content='How to improve your performance in an interview \nHere are some tips to improve your performance in an interview: \n1. Prepare an introduction \nAn introduction may be the first thing an interviewer asks. It is a great opportunity for \nthem to evaluate you and for you to highlight your skills. Prepare an introduction \nbeforehand and practice it. Keep it short, for instance, anywhere between 30 seconds \nand two minutes. Give a brief overview of your skills, experience and how you may be a \ngood fit for the job. \n2. Research the company \nBefore any interview, try to understand more abo

In [ ]:
### Text Spillting get into chunks

In [14]:

### Text splitting get into chunks

def document_splitting(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    # here the split_document is libraray function of RecursiveCharacterTextSpillte
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [16]:
chunks=document_splitting(all_pdf_documents)
chunks

Split 19 documents into 43 chunks

Example chunk:
Content: How to improve your performance in an interview 
Here are some tips to improve your performance in an interview: 
1. Prepare an introduction 
An introduction may be the first thing an interviewer asks...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-12-14T04:29:50+00:00', 'author': 'Huma  Iqbal', 'moddate': '2022-12-14T04:29:50+00:00', 'source': '..\\data\\pdf\\Ensuring Success in Job Interviews.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Ensuring Success in Job Interviews.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-12-14T04:29:50+00:00', 'author': 'Huma  Iqbal', 'moddate': '2022-12-14T04:29:50+00:00', 'source': '..\\data\\pdf\\Ensuring Success in Job Interviews.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Ensuring Success in Job Interviews.pdf', 'file_type': 'pdf'}, page_content='How to improve your performance in an interview \nHere are some tips to improve your performance in an interview: \n1. Prepare an introduction \nAn introduction may be the first thing an interviewer asks. It is a great opportunity for \nthem to evaluate you and for you to highlight your skills. Prepare an introduction \nbeforehand and practice it. Keep it short, for instance, anywhere between 30 seconds \nand two minutes. Give a brief overview of your skills, experience and how you may be a \ngood fit for the job. \n2. Research the company \nBefore any interview, try to understand more abo

In [ ]:
#  using inbuilt function for chunking 
"""     
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load everything in one shot
# glob="**/*.pdf" finds all PDFs in all subfolders
loader = DirectoryLoader("../data", glob="**/*.pdf", loader_cls=PyPDFLoader)
all_docs = loader.load()

# 2. Split everything in one shot
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(all_docs)

print(f"Total chunks created: {len(chunks)}")

 """

### Embedding and vectorDB

In [1]:

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity




d:\RAG-oneshot_KN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Creating Embedding 

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        # getting model name
        self.model_name = model_name
        # making model none which later be insert in this 
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            # this download hte model from huggingface
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


d:\RAG-oneshot_KN\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abdul\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4671.68it/s]
BertModel LOAD REPORT from: sentence-

Model loaded successfully. Embedding dimension: 384


In [ ]:
'''''
Persistn directory -
 Folder path where the vector database is saved permanently to the hard drive.
 

 ----------------------------------------------------------------------------

 . What actually happens inside the VectorDB?
When you call .add() in your code, three things happen simultaneously:

The Identity: Your Array is linked to a unique ID.

The Storage: The Array is saved to the disk so it survives a reboot.

The Indexing: The database calculates where this array sits "mathematically" compared to all the others. It draws "connections" to its nearest neighbors.
 
-------------------------------------------------------------------------------

how data are stored in collection 

{
    "ids": ["id1", "id2"],
    "embeddings": [[0.1, 0.2...], [0.1, 0.3...]],
    "documents": ["Text chunk 1", "Text chunk 2"],
    "metadatas": [{"source": "pdf1"}, {"source": "pdf1"}]
}
 
 '''